In [64]:
import matplotlib.pyplot as plt
import numpy as np 
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import linear_model
import pandas as pd
np.random.seed(0)

In [65]:
transform = transforms.ToTensor()
print("Đang tải tập Train...")
train_dataset = datasets.MNIST(root='../data',     
                               train=True,          
                               download=True,      
                               transform=transform)
print("Đang tải tập Test...")
test_dataset = datasets.MNIST(root='../data', 
                              train=False,    
                              download=True, 
                              transform=transform)
print(f"Hoàn tất! Số lượng ảnh train: {len(train_dataset)}")
print(f"Hoàn tất! Số lượng ảnh test: {len(test_dataset)}")
print(f"Kích thước ảnh train: {train_dataset.data.shape}")
print(f"Kích thước ảnh test: {test_dataset.data.shape}")

Đang tải tập Train...
Đang tải tập Test...
Hoàn tất! Số lượng ảnh train: 60000
Hoàn tất! Số lượng ảnh test: 10000
Kích thước ảnh train: torch.Size([60000, 28, 28])
Kích thước ảnh test: torch.Size([10000, 28, 28])


In [66]:
projection_matrix = np.random.rand(28*28, 128) * 0.01

def preprocess(tensor_data):
    # Flatten ảnh từ (N, 28, 28) thành (N, 784)
    X_numpy = tensor_data.data.numpy().reshape(-1, 28*28)
    X_numpy = X_numpy / 255.0
    y_numpy = tensor_data.targets.numpy()
    
    # Giảm chiều xuống còn 128
    X_projected = X_numpy
    return X_projected, y_numpy

train_X, train_y = preprocess(train_dataset)
test_X, test_y = preprocess(test_dataset)

In [67]:
def add_bias(X):
    n_samples = X.shape[0]
    bias = np.ones((n_samples, 1))
    return np.hstack((X, bias))

X_train_bias = add_bias(train_X)
X_test_bias = add_bias(test_X)

d = train_X.shape[1] 
W_bias = np.random.rand(d + 1, 10) 

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def grad_entropy_loss(X, y, W):
    n_samples = X.shape[0]
    z = np.dot(X, W)
    y_pred = softmax(z)
    
    # One-hot encoding
    y_one_hot = np.zeros_like(y_pred)
    y_one_hot[np.arange(n_samples), y] = 1
    
    # Tính gradient
    grad_W = np.dot(X.T, (y_pred - y_one_hot)) / n_samples
    return grad_W

def train(X, y, W, learning_rate=0.01, n_epochs=30, batch_size=64):
    N = X.shape[0]
    for epoch in range(n_epochs):
        indices = np.random.permutation(N) 
        X_shuffled = X[indices]
        y_shuffled = y[indices]
        
        for i in range(0, N, batch_size):
            X_batch = X_shuffled[i:i + batch_size]
            y_batch = y_shuffled[i:i + batch_size]
            
            grad_W = grad_entropy_loss(X_batch, y_batch, W)
            W -= learning_rate * grad_W
            
        print(f"Đã hoàn thành Epoch {epoch + 1}/{n_epochs}")
    return W

def predict(X, W):
    z = np.dot(X, W)
    y_pred = softmax(z)
    return np.argmax(y_pred, axis=1)

In [71]:
W_trained = train(X_train_bias, train_y, W_bias, learning_rate=0.01, n_epochs=10)
predictions = predict(X_test_bias, W_trained)
accuracy = np.mean(predictions == test_y)
print(f"Độ chính xác trên tập Test: {accuracy * 100:.2f}%")

Đã hoàn thành Epoch 1/10
Đã hoàn thành Epoch 2/10
Đã hoàn thành Epoch 3/10
Đã hoàn thành Epoch 4/10
Đã hoàn thành Epoch 5/10
Đã hoàn thành Epoch 6/10
Đã hoàn thành Epoch 7/10
Đã hoàn thành Epoch 8/10
Đã hoàn thành Epoch 9/10
Đã hoàn thành Epoch 10/10
Độ chính xác trên tập Test: 90.86%


In [70]:
import time
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("=== BẮT ĐẦU HUẤN LUYỆN VỚI SCIKIT-LEARN ===")

sklearn_model = LogisticRegression(
    solver='saga', 
    max_iter=10, 
    random_state=42,
    n_jobs=-1 
)

start_time = time.time()
sklearn_model.fit(train_X, train_y)
end_time = time.time()

print(f"Thời gian huấn luyện (Sklearn): {end_time - start_time:.4f} giây")

y_pred_train = sklearn_model.predict(train_X)
y_pred_test = sklearn_model.predict(test_X)

acc_train = accuracy_score(train_y, y_pred_train)
acc_test = accuracy_score(test_y, y_pred_test)

print(f"Độ chính xác trên tập Train: {acc_train * 100:.2f}%")
print(f"Độ chính xác trên tập Test : {acc_test * 100:.2f}%")

=== BẮT ĐẦU HUẤN LUYỆN VỚI SCIKIT-LEARN ===


d:\projects\ml-lab\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Thời gian huấn luyện (Sklearn): 11.0537 giây
Độ chính xác trên tập Train: 93.29%
Độ chính xác trên tập Test : 92.60%


d:\projects\ml-lab\venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
